In [0]:
df = spark.read.parquet("/Volumes/workspace/default/matrica/bronze/vct_2025/agents/agents_pick_rates")
df.printSchema()
df.show(5, truncate=False)

root
 |-- Tournament: string (nullable = true)
 |-- Stage: string (nullable = true)
 |-- Match Type: string (nullable = true)
 |-- Map: string (nullable = true)
 |-- Agent: string (nullable = true)
 |-- Pick Rate: string (nullable = true)

+-----------------------+--------+-------------------+--------+-----+---------+
|Tournament             |Stage   |Match Type         |Map     |Agent|Pick Rate|
+-----------------------+--------+-------------------+--------+-----+---------+
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|omen |95%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|yoru |60%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|viper|55%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|sova |55%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|vyse |50%      |
+-----------------------+--------+-------------------+--------+-----+---------+
only showing top 5 rows


In [0]:
df = spark.read.parquet("/Volumes/workspace/default/matrica/bronze/vct_2025/agents/maps_stats")
df.printSchema()
df.show(5, truncate=False)

root
 |-- Tournament: string (nullable = true)
 |-- Stage: string (nullable = true)
 |-- Match Type: string (nullable = true)
 |-- Map: string (nullable = true)
 |-- Total Maps Played: integer (nullable = true)
 |-- Attacker Side Win Percentage: string (nullable = true)
 |-- Defender Side Win Percentage: string (nullable = true)

+-----------------------+--------+-------------------+--------+-----------------+----------------------------+----------------------------+
|Tournament             |Stage   |Match Type         |Map     |Total Maps Played|Attacker Side Win Percentage|Defender Side Win Percentage|
+-----------------------+--------+-------------------+--------+-----------------+----------------------------+----------------------------+
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|10               |62%                         |38%                         |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|Lotus   |3                |56%                        

In [0]:
df = spark.read.parquet("/Volumes/workspace/default/matrica/bronze/vct_2025/agents/teams_picked_agents")
df.printSchema()
df.show(5, truncate=False)

root
 |-- Tournament: string (nullable = true)
 |-- Stage: string (nullable = true)
 |-- Match Type: string (nullable = true)
 |-- Map: string (nullable = true)
 |-- Team: string (nullable = true)
 |-- Agent: string (nullable = true)
 |-- Total Wins By Map: integer (nullable = true)
 |-- Total Loss By Map: integer (nullable = true)
 |-- Total Maps Played: integer (nullable = true)

+-----------------------+--------+-------------------+-----+------+-------+-----------------+-----------------+-----------------+
|Tournament             |Stage   |Match Type         |Map  |Team  |Agent  |Total Wins By Map|Total Loss By Map|Total Maps Played|
+-----------------------+--------+-------------------+-----+------+-------+-----------------+-----------------+-----------------+
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|Haven|FNATIC|omen   |1                |0                |1                |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|Haven|FNATIC|yoru   |1                |0 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ==========================================================
# PATHS
# ==========================================================

bronze_path = "/Volumes/workspace/default/matrica/bronze/vct_2025/agents/agents_pick_rates"
silver_path = "/Volumes/workspace/default/matrica/silver/vct_2025/agents/agents_pick_rates"

# ==========================================================
# READ PARQUET
# ==========================================================

df = spark.read.parquet(bronze_path)

print("=" * 120)
print("PROCESSING DATASET : agents_pick_rates")
print("=" * 120)

rows_before = df.count()

print(f"\nRows Before Cleaning : {rows_before}")
print(f"Columns : {len(df.columns)}")

print("\nSchema")
df.printSchema()

print("\nFirst 10 Records")
df.show(10, truncate=False)

# ==========================================================
# DUPLICATE COUNT
# ==========================================================

duplicates = rows_before - df.dropDuplicates().count()

print(f"\nDuplicate Rows : {duplicates}")

# ==========================================================
# NULL COUNT
# ==========================================================

print("\nNull Count")

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show(truncate=False)

# ==========================================================
# BLANK STRING COUNT
# ==========================================================

print("\nBlank String Count")

for c, dtype in df.dtypes:

    if dtype == "string":

        blank = df.filter(trim(col(c)) == "").count()

        print(f"{c} : {blank}")

# ==========================================================
# DATA CLEANING
# ==========================================================

# Remove duplicate rows
df = df.dropDuplicates()

# Remove rows where all columns are NULL
df = df.na.drop(how="all")

# Trim all string columns
for c, dtype in df.dtypes:

    if dtype == "string":

        df = df.withColumn(c, trim(col(c)))

# Replace empty strings with NULL
df = df.replace("", None)

# ==========================================================
# MAINTAIN CONSISTENT GRANULARITY
# ==========================================================

df = df.withColumn(
    "Stage",
    when(
        lower(col("Stage")) == "all stages",
        None
    ).otherwise(col("Stage"))
)

df = df.withColumn(
    "Match Type",
    when(
        lower(col("Match Type")) == "all match types",
        None
    ).otherwise(col("Match Type"))
)

df = df.withColumn(
    "Map",
    when(
        lower(col("Map")) == "all maps",
        None
    ).otherwise(col("Map"))
)

# ==========================================================
# REMOVE AGGREGATE ROWS
# ==========================================================

df = df.filter(col("Stage").isNotNull())
df = df.filter(col("Match Type").isNotNull())
df = df.filter(col("Map").isNotNull())

# ==========================================================
# STANDARDIZE AGENT NAMES
# ==========================================================

df = df.withColumn(
    "Agent",
    initcap(lower(col("Agent")))
)

# ==========================================================
# CONVERT PICK RATE
# 95% -> 0.95
# ==========================================================

df = df.withColumn(
    "Pick Rate",
    (
        regexp_replace(col("Pick Rate"), "%", "")
        .cast("double") / 100
    )
)

# ==========================================================
# STANDARDIZE COLUMN NAMES
# ==========================================================

for c in df.columns:

    new_name = (
        c.lower()
         .replace(" ", "_")
         .replace("-", "_")
    )

    df = df.withColumnRenamed(c, new_name)

# ==========================================================
# AFTER CLEANING
# ==========================================================

rows_after = df.count()

print(f"\nRows After Cleaning : {rows_after}")

print(f"Rows Removed : {rows_before - rows_after}")

print("\nUpdated Schema")

df.printSchema()

print("\nFirst 10 Cleaned Records")

df.show(10, truncate=False)

# ==========================================================
# WRITE TO SILVER
# ==========================================================

(
    df.write
      .mode("overwrite")
      .parquet(silver_path)
)

print("\nSaved To")

print(silver_path)

print("=" * 120)
print("Bronze → Silver ETL Completed Successfully")
print("=" * 120)

PROCESSING DATASET : agents_pick_rates

Rows Before Cleaning : 30294
Columns : 6

Schema
root
 |-- Tournament: string (nullable = true)
 |-- Stage: string (nullable = true)
 |-- Match Type: string (nullable = true)
 |-- Map: string (nullable = true)
 |-- Agent: string (nullable = true)
 |-- Pick Rate: string (nullable = true)


First 10 Records
+-----------------------+--------+-------------------+--------+------+---------+
|Tournament             |Stage   |Match Type         |Map     |Agent |Pick Rate|
+-----------------------+--------+-------------------+--------+------+---------+
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|omen  |95%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|yoru  |60%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|viper |55%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|sova  |55%      |
|Valorant Champions 2025|Playoffs|Upper Quarterfinals|All Maps|vyse  |50%      |
|Valo